<center>
<img src="https://upload.wikimedia.org/wikipedia/fr/thumb/1/1d/Logo_T%C3%A9l%C3%A9com_SudParis.svg/1014px-Logo_T%C3%A9l%C3%A9com_SudParis.svg.png" width="10%" />
</center>

<center> <h2> NET 4103/7431 Complex Network </h2> </center>

<center> <h3> Vincent Gauthier (vincent.gauthier@telecom-sudparis.eu) </h3> </center>

### Note
Avant de commencer les exercices, assurez-vous que tout fonctionne comme prévu. Tout d'abord, le redémarrage du kernel **(dans la barre de menus, sélectionnez le kernel $\rightarrow$ Restart)**.

Assurez-vous que vous remplir les célluler aux endroits marquer «YOUR CODE HERE». 

Veuillez supprimer les ligne «raise NotImplementedError()» dans toutes les cellules auxquelles vous avez répondu, ainsi que votre nom et prénom ci-dessous:

In [ ]:
NOM = "XXX"
PRENOM = "XXX"

---

# TF-IDF from scratch

This short tutorial implements term frequency–inverse document frequency (TF-IDF) using only Python's standard library. No third-party packages are required.

TF-IDF gives higher weight to terms that are frequent in one document but uncommon across the collection. It is a simple way to represent documents numerically for search and comparison.


## 1. Prepare a small document collection

Each string is one document. For simplicity, we lowercase text and split on whitespace; this basic tokenizer does not handle punctuation, contractions, or linguistic normalization.


In [ ]:
documents = [
    "cats chase small mice",
    "dogs chase cats",
    "mice eat small seeds",
]

# Lowercasing ensures that words with different capitalization match.
tokenized_documents = [document.lower().split() for document in documents]
for index, tokens in enumerate(tokenized_documents):
    print(f"Document {index}: {tokens}")


## 2. Build a vocabulary and count terms

The vocabulary is the set of unique terms in the collection. A term count records how often each vocabulary term occurs in each document.


In [ ]:
# Sort the vocabulary to make the output order predictable.
vocabulary = sorted({term for tokens in tokenized_documents for term in tokens})

# Build one term-count dictionary per document.
term_counts = []
for tokens in tokenized_documents:
    counts = {term: 0 for term in vocabulary}
    for term in tokens:
        counts[term] += 1
    term_counts.append(counts)

print("Vocabulary:", vocabulary)
print("Counts in document 0:", term_counts[0])


## 3. Compute term frequency (TF)

Here, TF is the count of a term divided by the total number of tokens in that document. Thus, TF values in each document sum to 1 (for a non-empty document). Other TF conventions exist.


In [ ]:
term_frequencies = []
for tokens, counts in zip(tokenized_documents, term_counts):
    document_length = len(tokens)
    # Avoid division by zero if an empty document is added.
    frequencies = {
        term: counts[term] / document_length if document_length else 0.0
        for term in vocabulary
    }
    term_frequencies.append(frequencies)

print("TF('cats') in document 0:", term_frequencies[0]["cats"])


## 4. Compute inverse document frequency (IDF)

Document frequency (DF) counts how many documents contain a term. This tutorial uses the smoothed formula `idf(t) = log((1 + N) / (1 + df(t))) + 1`, where `N` is the number of documents. Smoothing avoids division by zero and gives a finite IDF for every term. The logarithm comes from Python's built-in `math` module.


In [ ]:
from math import log

number_of_documents = len(tokenized_documents)
document_frequencies = {
    term: sum(1 for counts in term_counts if counts[term] > 0)
    for term in vocabulary
}

inverse_document_frequencies = {
    term: log((1 + number_of_documents) / (1 + document_frequencies[term])) + 1
    for term in vocabulary
}

for term in vocabulary:
    print(f"{term:>8}: df={document_frequencies[term]}, idf={inverse_document_frequencies[term]:.3f}")


## 5. Combine TF and IDF

The TF-IDF weight is `TF(t, d) × IDF(t)`. A term gets a high weight when it occupies a large share of a document and appears in relatively few documents overall.


In [ ]:
tf_idf_vectors = []
for frequencies in term_frequencies:
    # Keep the same vocabulary order in every vector.
    vector = {
        term: frequencies[term] * inverse_document_frequencies[term]
        for term in vocabulary
    }
    tf_idf_vectors.append(vector)

# Display non-zero weights for each document.
for index, vector in enumerate(tf_idf_vectors):
    print(f"\nDocument {index}: {documents[index]}")
    for term, weight in vector.items():
        if weight > 0:
            print(f"  {term:>8}: {weight:.3f}")


## 6. Use TF-IDF to compare documents

Cosine similarity compares vector directions and is often used with TF-IDF. We implement the dot product and vector lengths directly. An empty or all-zero vector has similarity 0 by convention.


In [ ]:
from math import sqrt

def cosine_similarity(vector_a, vector_b):
    """Return cosine similarity for two term-to-weight dictionaries.

    Missing terms are treated as having weight zero. Returns 0.0 if either
    vector has zero length.
    """
    dot_product = sum(vector_a[term] * vector_b[term] for term in vocabulary)
    norm_a = sqrt(sum(vector_a[term] ** 2 for term in vocabulary))
    norm_b = sqrt(sum(vector_b[term] ** 2 for term in vocabulary))
    if norm_a == 0 or norm_b == 0:
        return 0.0
    return dot_product / (norm_a * norm_b)

for i in range(len(documents)):
    for j in range(i + 1, len(documents)):
        score = cosine_similarity(tf_idf_vectors[i], tf_idf_vectors[j])
        print(f"Documents {i} and {j}: cosine similarity = {score:.3f}")


## Key takeaways and limitations

- TF emphasizes terms frequent within a document; IDF downweights terms shared by many documents.
- TF-IDF vectors depend on the chosen document collection: adding documents can change IDF values.
- This simple tokenizer treats `cats` and `cats,` as different terms and does not understand synonyms or word meaning.
- The implementation uses plain Python dictionaries and standard-library `math`; no external library is needed.

### Try it yourself

Add a fourth document about a new topic. Recompute the counts, TF, IDF, and vectors, then observe how the IDF scores change. What happens to a term that appears in every document?
